In [ ]:
import os
import openpyxl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm  # 导入tqdm库
from openpyxl.utils.exceptions import InvalidFileException

# 解决中文显示问题
font_path = 'C:/Windows/Fonts/simsun.ttc'
import matplotlib
matplotlib.rcParams['font.family'] = 'SimSun'  # 指定字体名称
matplotlib.rcParams['font.sans-serif'] = ['SimSun']  # 指定字体名称（以防fallback）
matplotlib.rcParams['axes.unicode_minus'] = False  # 正确显示负号

# 获取"charge2"文件夹下符合"fixed_data*_charge*"模式的所有Excel文件
path = "./charge2"  # 指定文件夹路径
files = [f for f in os.listdir(path) if f.startswith('fixed_data') and 'charge' in f and f.endswith('.xlsx')]

# 存储每个文件的估计结果
results = []

# 使用tqdm包装文件列表，显示处理进度
for file in tqdm(files, desc="处理文件", unit="文件"):  # 添加进度条
    try:
        # 尝试打开文件
        file_path = os.path.join(path, file)
        work = openpyxl.load_workbook(file_path)  # 读取Excel文件
        sheet = work.active

        # 读取电压和电流数据，从第二行开始检索
        voltage = [cell.value for col in sheet.iter_cols(min_col=5, max_col=5, min_row=2) for cell in col]
        current = [cell.value for col in sheet.iter_cols(min_col=6, max_col=6, min_row=2) for cell in col]

        # 转换为数值，去除任何非数值行
        voltage = [v for v in voltage if isinstance(v, (int, float))]
        current = [c for c in current if isinstance(c, (int, float))]

        # 初始化参数
        V_oc = 580  # 初始开路电压
        a0 = 0.99
        b0 = 1
        b1 = 1
        theta = np.array([a0, b0, b1])  # 参数向量
        P = np.eye(3) * 1000  # 初始协方差矩阵
        lambda_ = 0.98  # 遗忘因子（接近1，强调最新数据）
        delt_t = 10  # 采样时间
        V_est = []  # 存放估计电压
        E_pre = []
        DE = []
        # 存储估计结果
        estimates = []
        error = []
        neizu = []
        jihuaneizu = []
        jihuadianrong = []

        # 递推最小二乘法
        for k in range(20, len(current)-1):  # 从第20个采样点开始
            phi = np.array([voltage[k], current[k+1], current[k]], dtype=np.float64)  # 构建设计矩阵
            V_pred = np.dot(theta, phi)  # 计算预测值
            V_est.append(V_pred)

            # 计算误差
            e = voltage[k+1] - V_pred
            new_e = e / voltage[k+1] * 100
            error.append(new_e)

            # 更新协方差矩阵
            K = P @ phi / (lambda_ + phi.T @ P @ phi)
            P = (P - np.outer(K, phi.T @ P)) / lambda_

            # 更新参数
            theta = theta + K * e

            # 存储估计结果
            R = -1 * (theta[1] - theta[2]) / (1 + theta[0])
            R_P = (theta[2] + theta[1]) / (1 + theta[0]) - (theta[1] - theta[2]) / (1 + theta[0])
            C_p = (1 + theta[0]) / (2 * R_P * (1 + theta[0]))
            list = np.array([R, R_P, C_p])
            estimates.append(list)
            neizu.append(R)
            jihuaneizu.append(R_P)
            jihuadianrong.append(abs(C_p))

        # 计算每个文件的内阻、极化电阻、极化电容的平均值（取20个采样点后的平均值）
        avg_R = np.mean(neizu[-20:])
        avg_RP = np.mean(jihuaneizu[-20:])
        avg_CP = np.mean(jihuadianrong[-20:])
        
        # 将文件名和估计结果添加到结果列表
        results.append([file, avg_R, avg_RP, avg_CP])

    except InvalidFileException:
        print(f"文件 {file} 无效，跳过此文件。")
        continue  # 跳过此文件并继续处理下一个文件
    except Exception as e:
        print(f"处理文件 {file} 时出现错误: {e}")
        continue  # 跳过此文件并继续处理下一个文件

# 将结果保存到"charge2"文件夹中
output_path = "./charge2/估计结果.xlsx"  # 设置输出路径到charge2文件夹
df = pd.DataFrame(results, columns=["文件名", "内阻 (Ω)", "极化电阻 (Ω)", "极化电容 (F)"])

# 检查空缺值，并用临近值填补
df.fillna(method='ffill', inplace=True)  # 用前一个有效值填补
df.fillna(method='bfill', inplace=True)  # 如果前一个有效值不可用，用后一个有效值填补

df.to_excel(output_path, index=False)

# 绘制内阻、极化电阻、极化电容的变化曲线图
plt.figure(figsize=(12, 6))

# 绘制内阻曲线
plt.subplot(1, 3, 1)
plt.plot(range(1, len(df) + 1), df["内阻 (Ω)"], color='red', label="内阻 (Ω)")
plt.xlabel('索引')
plt.ylabel('内阻 (Ω)')
plt.title('内阻变化曲线')

# 绘制极化电阻曲线
plt.subplot(1, 3, 2)
plt.plot(range(1, len(df) + 1), df["极化电阻 (Ω)"], color='blue', label="极化电阻 (Ω)")
plt.xlabel('索引')
plt.ylabel('极化电阻 (Ω)')
plt.title('极化电阻变化曲线')

# 绘制极化电容曲线
plt.subplot(1, 3, 3)
plt.plot(range(1, len(df) + 1), df["极化电容 (F)"], color='green', label="极化电容 (F)")
plt.xlabel('索引')
plt.ylabel('极化电容 (F)')
plt.title('极化电容变化曲线')

plt.tight_layout()
plt.show()
